# 回测基本框架

将价格、测试因子和分析节点组成计算图，再由引擎返回统计结果。需要报告时接入对应的报告节点；需要持久化时接入 `BTStorer`。

下面以 IC 测试演示“准备数据 → 构造分析 → 获取报告 → 保存结果”。示例在临时目录中生成确定性数据，无需外部数据库。

## 完整示例：IC 测试

下面展示一个使用回测框架进行 IC 测试的完整示例。

In [ ]:
import datetime as dt
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from QuantStudio.Core.QSObject import Panel
from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.Tools.Visualization import qs_help

In [ ]:
# 在临时目录中创建独立的演示库，所有数据由固定随机种子生成。
demo_directory = TemporaryDirectory()
demo_root = Path(demo_directory.name)
(demo_root / "Cache").mkdir()
factor_dir = demo_root / "HDF5"
factor_dir.mkdir()
FDB = HDF5DB(args={"MainDir": factor_dir}).connect()
rng = np.random.default_rng(0)
demo_dates = list(pd.date_range("2025-01-01", "2025-04-30").to_pydatetime())
demo_ids = [f"{i:06d}.SZ" for i in range(1, 16)]
demo_shape = (len(demo_dates), len(demo_ids))
demo_price = pd.DataFrame(
    10 * np.exp(np.cumsum(rng.normal(.001, .01, demo_shape), axis=0)),
    index=demo_dates, columns=demo_ids,
)
FDB.writeData(Panel({"close": demo_price, "open": demo_price * .999,
                     "high": demo_price * 1.01, "low": demo_price * .99}),
              table_name="stock_cn_day_bar")
FDB.writeData(Panel({"if_listed": pd.DataFrame(1., index=demo_dates, columns=demo_ids)}),
              table_name="stock_cn_status")
FDB.writeData(Panel({"industry": pd.DataFrame(
    np.tile(["TMT", "Ind", "Fin"] * 5, (len(demo_dates), 1)),
    index=demo_dates, columns=demo_ids)}),
    table_name="stock_cn_industry", data_type={"industry": "string"})
FDB.writeData(Panel({name: pd.DataFrame(rng.uniform(.01, .2, demo_shape),
                                      index=demo_dates, columns=demo_ids)
                    for name in ("bp_lr", "ep_ttm")}), table_name="stock_cn_factor_value")
FDB.writeData(Panel({"close": demo_price.mean(axis=1).to_frame("000905.SH")}),
              table_name="index_cn_day_bar")

StartDT, EndDT = dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 31)# 数据起止时间
TestStartDT, TestEndDT = dt.datetime(2025, 2, 28), EndDT# 测试起止时间

FT = FDB.getTable("stock_cn_day_bar")
DTRuler = FT.getDateTime(start_dt=StartDT, end_dt=EndDT)
TestDTs = FT.getDateTime(start_dt=TestStartDT, end_dt=TestEndDT)
SectionIDs = IDs = FT.getID()

# 再平衡时点序列
from QuantStudio.Tools.DateTimeFun import getMonthLastDateTime
BalanceDTs = getMonthLastDateTime(DTRuler)# 月末

In [ ]:
# 构建回测
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.BackTest.BackTestModel import BTReport, BTNode
from QuantStudio.BackTest.SectionFactor.IC import CalcIC, IC, ICReport

# 获取数据
FT = FDB.getTable("stock_cn_status")
Mask = (FT.getFactor("if_listed")==1)

FT = FDB.getTable("stock_cn_day_bar")
Price = FT.getFactor("close")

FT = FDB.getTable("stock_cn_factor_value")
FactorList = [FT.getFactor(iFactorName) for iFactorName in ["bp_lr", "ep_ttm"]]

# 步骤1: 创建因子算子 (计算 IC 因子)
FactorIC = CalcIC(
    descriptor_ids=SectionIDs,
    lookback=31,            # 在时间标尺上的回溯期数
    period_lookback=1,      # 在计算标尺上的回溯期数（月度IC, period_lookback=1表示用上月因子值）
    corr_method="spearman"  # 相关性方法: spearman / pearson / kendall
)(
    *FactorList,            # 待测试的因子列表
    price=Price,            # 价格因子（用于计算收益率）
    mask=Mask,              # 筛选条件（仅在上市状态为1时计算）
    factor_args={"CalcDTRuler": BalanceDTs}  # 按月计算 IC
)

# 步骤2: 创建回测计算节点
ICNode = IC(
    FactorIC,
    args={
        "Name": "IC测试",
        "RollingAvgPeriod": 2,  # IC 移动平均期数
    }
)

# 步骤3: 创建报告节点
ICReportNode = ICReport(ICNode)

# 步骤4: 创建报告汇总容器并执行
Report = BTReport(report_nodes=[ICReportNode])

CacheDir = demo_root / "Cache"
if not CacheDir.exists(): CacheDir.mkdir(parents=True)

with FeatherFactorCache(args={"DTRuler": DTRuler, "PIDs": ["0"], "CacheDir": CacheDir, "StartMode": "new"}) as Cache:
    with FactorContext(PID="0", PIDList=["0"], DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Output, = ExecEngine.run(
                [Report],
                Context,
                fwd_data_list=[DTLocalContext(DTs=TestDTs)]
            )

# 步骤5: 查看报告

print(Output["0-" + ICReportNode.Name]["统计数据"])

In [ ]:
from IPython.display import HTML, display

display(HTML(Output["Report"]))

## 调用流程与结果

构造 `CalcIC` 等算子得到中间因子，交给 `IC` 等统计节点。统计结果可以直接由引擎读取；需要 HTML 时加上 `ICReport`，多个报告用 `BTReport(report_nodes=[...])` 汇总。

`Engine.run()` 返回与目标节点顺序一致的列表。`FactorContext` 设置完整 `DTRuler` 和 `SectionIDs`，`DTLocalContext(DTs=...)` 指定本次回测日期。使用磁盘缓存时先启动缓存，再将其赋给 `DataCache`。

返回字典的 `Report` 保存汇总 HTML，`"0-" + ICReportNode.Name` 保存第一个报告节点的数据。缓存接入见 [缓存](../Core/缓存.ipynb)，扩展节点的生命周期见 [原理和细节](原理和细节.ipynb)。

## 子模块

回测框架按功能分为四个子模块，每个子模块有独立的文档：

| 子模块 | 说明 | 详细文档 |
|--------|------|----------|
| `SectionFactor` | 截面因子测试：IC 分析、IC 衰减、分位数组合、因子换手率、截面相关性、Fama-MacBeth 回归 | [截面因子测试](截面因子测试.ipynb) |
| `Strategy` | 策略回测：账户创建、策略信号生成、净值与收益率计算、回测报告 | [策略回测](策略回测.ipynb) |
| `PerformanceAnalysis` | 绩效归因：Brinson 模型分解超额收益来源 | [业绩归因](业绩归因.ipynb) |
| `Risk` | 风险模型测试：Bias 检验验证风险预测准确性 | [风险模型测试](风险模型测试.ipynb) |
| 结果持久化 | 回测结果存储与管理：`BTResultDB` 接口、HDF5 实现、`BTStorer` 节点 | [BTStorer](BTStorer.ipynb) |

<a id="result-storage"></a>

## 回测结果持久化

`BTNode` / `BTReport` 的输出（嵌套 dict，叶节点为 DataFrame、Series、str、float 等）默认仅存在于内存中。通过 `BTStorer` 节点和 `BTResultDB` 结果库，可以将回测结果持久化到磁盘。

### 分层架构

回测结果持久化采用与因子框架相同的分层模式：

```
BTResultDB (抽象基类, 定义读写接口)
  └── HDF5BTResultDB   (目录模式 HDF5 实现, 每个结果组一个文件)

BTStorer (Node) — 插入计算图, 作为 BTNode/BTReport 的下游, 将结果写入 BTResultDB
```

### BTResultDB — 回测结果库

`BTResultDB` 定义了回测结果的读写接口：

| 方法 | 说明 |
|------|------|
| `writeResult(result, group_name, metadata=None)` | 写入一组回测结果 |
| `readResult(group_name)` | 按名称读取一组结果 |
| `listResults(metadata=None)` | 列出已存储的结果组, 支持按 metadata 筛选 |
| `readMetaData(group_name, key=None)` | 读取结果组的元信息 |
| `setMetaData(group_name, key=None, value=None, metadata=None)` | 设置结果组的元信息 |

`group_name` 支持路径层级（如 `"A股/IC/沪深300"`），`metadata` 是可选的标签 dict，用于后续查询筛选。

### BTStorer — 持久化节点

`BTStorer` 是一个 `Node`，插入计算图中作为 `BTReport` 的下游，将回测结果写入 `BTResultDB`。

**参数：**

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"BTStorer"` | 节点名称 |
| `TargetDB` | `BTResultDB` | 必填 | 目标结果库对象 |
| `GroupName` | `Optional[str]` | `None` | 父分组路径，支持路径层级；最终结果组为 `GroupName/依赖节点.Name`，`None` 或空字符串时直接使用依赖节点 Name |
| `Metadata` | `Optional[dict]` | `None` | 元信息标签, 透传给 `BTResultDB.writeResult` |

**两种接入方式：**

```python
# 方式 1: 存储整个 BTReport 的合并输出
Report = BTReport(report_nodes=[ICReportNode, AccountReportNode])
Storer = BTStorer(deps=[Report], args={"TargetDB": db, "GroupName": "A股/IC测试"})

# 方式 2: 存储单个 ReportNode 的输出
Storer = BTStorer(deps=[ICReportNode], args={"TargetDB": db, "Metadata": {"资产": "A股"}})
```

当 `BTStorer` 有多个依赖时，自动拆分为每个依赖一个子 Storer 实例（split 模式），解决并行 Engine 的写入冲突。

```
ReportNode1(ICReport) ─┐                          ├─→ BTStorer → ResultDB: A股/ICReport
ReportNode2(账户)  ────┼─→ BTStorer(GroupName="A股")─┼─→ BTStorer → ResultDB: A股/账户
ReportNode3(相关)  ────┘                          └─→ BTStorer → ResultDB: A股/相关
```

子 Storer 继承同一个 `GroupName`，各自追加依赖节点名称；`split=False` 时结果组路径也遵循相同规则。

**读取结果：**

读取结果及元信息时使用完整路径。例如方式 1 中存储的报告，结果组为 `A股/IC测试/{Report.Name}`：

```python
group_name = f"A股/IC测试/{Report.Name}"
result = db.readResult(group_name)
metadata = db.readMetaData(group_name)
all_groups = db.listResults(metadata={"资产": "A股"})
```

## 完整示例：回测结果持久化

下面展示如何使用 `BTStorer` 将上面的 IC 测试结果持久化到磁盘，并在后续读取使用。

In [ ]:
# 构建带持久化的回测: BTReport → BTStorer
from QuantStudio.BackTest.BTResultDB import HDF5BTResultDB
from QuantStudio.BackTest.BTStorer import BTStorer

# 创建结果库 (目录模式)
ResultDir = demo_root / "BTResults"
ResultDir.mkdir()
ResultDB = HDF5BTResultDB(args={"MainDir": str(ResultDir)})

# 复用上面的 ICNode 和 ICReportNode, 在 BTReport 之后接入 BTStorer
ICReportNode2 = ICReport(ICNode)
Report2 = BTReport(report_nodes=[ICReportNode2])
Storer = BTStorer(
    deps=[Report2],
    args={
        "TargetDB": ResultDB,
        "GroupName": "A股/IC测试",
        "Metadata": {"资产": "A股", "策略": "IC", "日期范围": f"{TestStartDT:%Y%m%d}-{TestEndDT:%Y%m%d}"},
    }
)

with FeatherFactorCache(args={"DTRuler": DTRuler, "PIDs": ["0"], "CacheDir": CacheDir, "StartMode": "new"}) as Cache:
    with FactorContext(PID="0", PIDList=["0"], DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Output2, = ExecEngine.run([Storer], Context, fwd_data_list=[DTLocalContext(DTs=TestDTs)])

print("写入完成")

In [ ]:
# 从结果库读取回测结果
ResultDB = HDF5BTResultDB(args={"MainDir": str(ResultDir)})

# 列出所有结果组
print("所有结果组:", ResultDB.ResultNames)

# 按 metadata 筛选
a_stock_results = ResultDB.listResults(metadata={"资产": "A股"})
print("A股结果:", a_stock_results)

# 读取结果
group_name = f"{Storer.Args.GroupName}/{Report2.Name}"
loaded = ResultDB.readResult(group_name)
print("\n结果 keys:", list(loaded.keys()))
print("\n统计数据:")
print(list(loaded))